> 基于摘要/summary(recurrent state) 的 linear attention

- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/08_deltanet/README.md
    -  Qwen3-Next vs. Kimi Linear 

### self-attention

经典 causal Self-Attention 会保留此前所有 token 的 Key 和 Value：

$$
K_{\le t}
=\begin{bmatrix}
k_1^\top\\
\vdots\\
k_t^\top
\end{bmatrix}
\in\mathbb{R}^{t\times d_k},
$$

$$
V_{\le t}
=\begin{bmatrix}
v_1^\top\\
\vdots\\
v_t^\top
\end{bmatrix}
\in\mathbb{R}^{t\times d_v}.
$$

当前 query \(q_t\) 与每个历史 key 做点积：
$$
a_t
=\operatorname{softmax}
\left(
\frac{K_{\le t}q_t}{\sqrt{d_k}}
\right)
\in\mathbb{R}^{t},
$$

再根据权重组合所有历史 value：
$$
o_t
=V_{\le t}^\top a_t.
$$

### GDN

$$
S_t=\alpha_t\odot S_{t-1}+\beta_t k_t\left(v_t-k_t^\top(\alpha_t\odot S_{t-1})\right)^\top 
$$

- 公式里的 $\alpha_t$ 就是 forget gate
    - 实现一般不直接预测 $\alpha_t$，而是预测其对数：$g_t=\log\alpha_t, \qquad \alpha_t=\exp(g_t).$

$$
\begin{aligned}z_t &= W_a^\text{up}W_a^\text{down}x_t,\\g_t &= -\operatorname{softplus}(z_t),\\\alpha_t &= \exp(g_t).\end{aligned}
$$

- $\operatorname{softplus}(z)\ge0\Rightarrow g_t\le0\Rightarrow0<\exp(g_t)\le1.$
    - 这样天然保证遗忘系数不会大于 1。
    - $\operatorname{softplus}(x)=\log(1+e^x).$，可以理解为 ReLU 的平滑版本：$\operatorname{ReLU}(x)=\max(0,x).$
- $\beta_t$ 是另一个门：写入门

| 门 | 数学量 | 作用 |
|---|---|---|
| 遗忘门（forget gate） | $\alpha_t=\exp(g_t)$ | 在写入前衰减旧状态 |
| 写入门（update/write gate） | $\beta_t$ | 控制当前 delta correction 强度 |

- GDN 的状态更新可以写成：
    - $\tilde S_{t-1}=\alpha_t S_{t-1},$
    $S_t=\tilde S_{t-1}+\beta_tk_t\left(v_t-\tilde S_{t-1}^\top k_t\right)^\top.$
- 输出：$o_t=S_t^\top q_t.$
- 在第 $t$ 个 token 更新完成之后，下一步只需要：$S_t\in\mathbb{R}^{d_k\times d_v}.$
    - 此前的：$k_1,v_1,\ldots,k_t,v_t$都可以丢掉。

```
当前状态 S_{t-1}
当前 token 的 k_t, v_t
          ↓
更新得到 S_t
          ↓
丢弃 k_t, v_t
          ↓
下一 token 只携带 S_t
```

GDN/KDA 在 prefill 中仍会产生：
```
q : [B, T, H, d_k]
k : [B, T, H, d_k]
v : [B, T, H, d_v]
```
因为 GPU 需要并行处理 prompt。
但经过 chunk reduction 之后，只保留：`S : [B, H, d_k, d_v]`

### details & analysis

$$
S_t\in\mathbb{R}^{d_k\times d_v}.
$$

- $d_k$：Key/Query 的特征维度，也就是“地址空间”维度；
- $d_v$：Value/Output 的特征维度，也就是“内容空间”维度；
- $S_t$：一个从地址空间映射到内容空间的线性映射，也可以看成固定大小的关联记忆。

```
q, k : [B, T, H, d_k]
v, o : [B, T, H, d_v]
S    : [B, H, d_k, d_v]
```

- $S_t$ 是一个 Key → Value 映射
    - $ S_t^\top\in\mathbb{R}^{d_v\times d_k}, \qquad q_t\in\mathbb{R}^{d_k}, $
    - $o_t=S_t^\top q_t\in\mathbb{R}^{d_v}.$

#### 显存分析

- 假设单 head：$d_k=d_v=128.$
    - GDN 状态大小：$d_kd_v=128\times128=16384$ 个数值
    - 经典 Attention 每个 token 的 KV cache：$d_k+d_v=128+128=256$ 个数值
    - GDN 固定状态大约相当于经典 Attention 的：$\frac{16384}{256}=64$ 个 token KV cache。
- 按 BF16、32 heads 计算：
    - GDN，每层固定状态 $32\times128\times128\times2=1{,}048{,}576\text{ bytes}\approx1\text{ MiB}.$
    - 标准 MHA，每层 1M token KV cache $10^6\times32\times(128+128)\times2\approx 15.3\text{ GB}.$